In [ ]:
# Define the neural network
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from src.neuralnet import NeuralNetwork

torch.set_default_dtype(torch.float32)  # Set global default dtype to float64

inputs = [[1.0, 2.0]]
outputs = [[.5, .75]]

layer_sizes = [2, 3, 4, 2]

X = np.array(inputs,dtype=np.float32)
Y = np.array(outputs, dtype=np.float32)

class SimpleNN(nn.Module):
    def __init__(self, layer_sizes):
        super(SimpleNN, self).__init__()
        self.layers = nn.ModuleList()
        for i in range(len(layer_sizes) - 1):
            self.layers.append(nn.Linear(layer_sizes[i], layer_sizes[i+1]))
        self.activation = nn.Sigmoid()
    
    def forward(self, x):
        for layer in self.layers:
            x = self.activation(layer(x))
        return x

# 

# initialize my network
net = NeuralNetwork(layer_sizes=layer_sizes)
w_grads = [np.zeros(matrix.shape) for matrix in net.weights]
b_grads = [np.zeros(vector.shape) for vector in net.bias]

# Initialize the torch_net
torch_net = SimpleNN(layer_sizes=layer_sizes)

with torch.no_grad():  # Disable gradient computation during assignment
    for i, layer in enumerate(torch_net.layers):
        layer.weight = nn.Parameter(torch.from_numpy(net.weights[i]).float())
        layer.bias = nn.Parameter(torch.from_numpy(net.bias[i]).float())

# Define the loss function and optimizer
criterion = nn.MSELoss()
optimizer = optim.SGD(torch_net.parameters(), lr=0.01)

# Forward pass
torch_output = torch_net(torch.from_numpy(X))

# Compute the loss
loss = criterion(torch_output, torch.from_numpy(Y))

# Backward pass
optimizer.zero_grad()
loss.backward()

# # Retrieve gradients
# print("Gradients:")
# for name, param in torch_net.named_parameters():
#     if param.grad is not None:
#         print(f"{name} - grad:\n{param.grad}")

for x, y in zip(X, Y):
    # print("x id:",id(x))
    z0 = net.weights[0] @ x + net.bias[0]
    zs = [z0]
    a0 = net.activation_f(z0)
    activations = [a0]
    for l in range(1, net.layer_n - 1, 1):
        # print("layers:",l,l-1)
        zl = net.weights[l] @ activations[l - 1] + net.bias[l]
        activation = net.activation_f(zl)
        # print("activation:", activation)
        zs.append(zl)
        activations.append(activation)

    z_output = zs[-1]
    a_output = activations[-1]
    output_error = net.cost_grad(a_output, y) * net.activation_df(
                    z_output
                )
    errors = [output_error]
    for l in range(net.hidden_layer_n, 0, -1):
        error = (net.weights[l].T @ errors[-1] * net.activation_df(zs[l - 1]))
        errors.append(error)

    errors.reverse()
                # compute sum of error
    for l in range(0, net.hidden_layer_n + 1, 1):
        w_grads[l] += np.outer(
                        errors[l], activations[l - 1] if l > 0 else x
                    )
                    # print(w_grads)
        b_grads[l] += errors[l]


# i = 0
# for (w_grad, b_grad) in zip(w_grads,b_grads):
#     print(f"layer {i} - grad:\n{w_grad, b_grad}")
#     i += 1


torch_grads = list(torch_net.named_parameters())
# Loop through the gradients and compare
for i in range(len(w_grads)):
    # Compare weight gradients
    assert np.allclose(w_grads[i], torch_grads[2 * i][1].grad.numpy()), f"Weight gradient mismatch at layer {i}"
    # Compare bias gradients
    assert np.allclose(b_grads[i], torch_grads[2 * i + 1][1].grad.numpy()), f"Bias gradient mismatch at layer {i}"

Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "/Users/dlakhdar/physics/repos/neural-network-demo/.neuralnet/lib/python3.12/site-packages/IPython/core/interactiveshell.py", line 3577, in run_code
  File "/var/folders/04/2cqfhcv133s3gbkf3b35tnkr0000gn/T/ipykernel_1131/1578497665.py", line 2, in <module>
    import numpy as np
ModuleNotFoundError: No module named 'numpy'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/Users/dlakhdar/physics/repos/neural-network-demo/.neuralnet/lib/python3.12/site-packages/pygments/styles/__init__.py", line 45, in get_style_by_name
ModuleNotFoundError: No module named 'pygments.styles.default'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/Users/dlakhdar/physics/repos/neural-network-demo/.neuralnet/lib/python3.12/site-packages/IPython/core/interactiveshell.py", line 2168, in showtraceback
  File "/Users/dlakhdar/physics/repos/neu